# Parameter Sensitivity Explorer

This notebook performs a **vectorized grid search** using the `ggTrader` orchestrator api. It visualizes the profitability landscape to find robust parameter regions.

In [37]:
import sys
import os
import pandas as pd
import numpy as np
import vectorbt as vbt
import plotly.graph_objects as go
from tabulate import tabulate

# Auto-reload custom modules
%load_ext autoreload
%autoreload 2

# Ensure project root is in path
project_root = os.path.abspath(os.path.join(os.getcwd(), '..',  'src'))
if project_root not in sys.path:
    sys.path.append(project_root)

from ggTrader.core.orchestrator import run_sensitivity_orchestrator

print("Environment initialized.")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Environment initialized.


In [38]:
# --- Configuration ---
CONSTANTS = {
    "SYMBOLS": None,
    "SYMBOLS_FILE": os.path.join(os.getcwd(), "..", "data", "top_20_USD_1095_movers.json"),
    "START_DATE": "2023-01-01",
    "END_DATE": "2025-12-31",
    "INTERVAL": "4h",
    "START_CASH": 10000,
    "PORTFOLIO_SHARE": 0.10,
    "FEES": 0.004,
    "MIN_TRADES": 10,
}

print("Configuration loaded.")

Configuration loaded.


In [39]:
# --- Define Parameter Grid ---
params = {
    "adx_threshold": list(range(25, 45, 5)),
    "adx_length": list(range(20, 40, 4)),
    "sar_acceleration": [0.02],
    "sar_maximum": [0.2],
    "atr_multiplier": list(np.arange(0.25, 3.1, 0.25)),
    "atr_length": list(range(14, 30, 4)),
    "use_dmp_cross": [True, False],
}

print("Parameter grid defined.")

Parameter grid defined.


In [40]:
# --- Run Vectorized Analysis ---
# Note: show_progress=True enables VectorBT's tqdm progress bar
results = run_sensitivity_orchestrator(
    config=CONSTANTS, param_grid=params, save_results=False, show_progress=True
)

results_df = results["results_df"]
best_params = results["best_params"]

print("\nAnalysis Complete.")

Loading data...
Running Vectorized Sensitivity Analysis...


  0%|          | 0/1920 [00:00<?, ?it/s]


Analysis Complete.


In [ ]:
top_10 = results_df.sort_values("Sharpe Ratio", ascending=False).head(10)
print("\nTOP 10 COMBINATIONS:")
print(tabulate(top_10, headers="keys", tablefmt="simple", showindex=False))

print("\nTOP PARAMETER STATS")
print(tabulate(top_10.describe(), headers="keys", tablefmt="simple", showindex=False))   


TOP 10 COMBINATIONS:
  adx_length    adx_threshold    sar_acceleration    sar_maximum  use_dmp_cross      atr_length    atr_multiplier    Sharpe Ratio
------------  ---------------  ------------------  -------------  ---------------  ------------  ----------------  --------------
          20               25                0.02            0.2  False                      18              0.25         2.31702
          20               25                0.02            0.2  False                      14              0.25         2.31121
          20               25                0.02            0.2  False                      26              0.25         2.30897
          20               25                0.02            0.2  False                      22              0.25         2.30457
          28               30                0.02            0.2  True                       26              0.5          2.29286
          28               25                0.02            0.2  Fa

In [42]:
# --- Visualization Helper ---
def show_heatmap(df, x_param, y_param, metric="Sharpe Ratio"):
    """Generates and displays a heatmap for the given parameter pair."""
    heatmap_data = df.pivot_table(
        index=y_param, 
        columns=x_param, 
        values=metric,
        aggfunc="mean"
    )

    fig = go.Figure(data=go.Heatmap(
        z=heatmap_data.values,
        x=heatmap_data.columns,
        y=heatmap_data.index,
        colorscale='Viridis',
        colorbar=dict(title=metric)
    ))

    fig.update_layout(
        title=f"{metric} Landscape: {y_param} vs {x_param}",
        xaxis_title=x_param,
        yaxis_title=y_param
    )

    fig.show()

print("Visualization helper defined.")

Visualization helper defined.


In [43]:
# --- Generate Heatmaps ---
# You can list any pairs of parameters you want to explore
pairs_to_plot = [
    ("adx_threshold", "adx_length"),
    # ("sar_acceleration", "sar_maximum"),
    ("atr_multiplier", "atr_length"),
    ("adx_length", "atr_length"),
    ("use_dmp_cross", "adx_length"),
    ("use_dmp_cross", "adx_threshold")
]

for x, y in pairs_to_plot:
    show_heatmap(results_df, x, y)